### Ćwiczenie 1

Utworzyć serię danych ciśnienia atmosferycznego w hPa w pierwszych siedmiu dniach roku 2024 w Warszawie. Wartości liczbowe można pobrać [stąd](https://www.ekologia.pl/pogoda/polska/mazowieckie/warszawa/archiwum,zakres,01-01-2024_07-01-2024) (przepisując ręcznie). (Zob. `period_range` i konstruktor `Series`).

In [ ]:
import pandas as pd
# przećwiczenie porzadkowania danych według dat 
cisnienie_dni = [
    ("01.10", 1022),
    ("02.10", 1021),
    ("03.10", 1014),
    ("04.10", 1022),
    ("05.10", 1018),
    ("06.10", 1023)
]


times = pd.period_range(cisnienie_dni[0][0], cisnienie_dni[-1][0])
pd.DataFrame([cisnienie_dni[i][1] for i in range(len(cisnienie_dni))], times)



### Ćwiczenie 2
Wczytaj dane z pliku *iris.csv*, oraz: 
- wypisz informacje o tych danych (jakie są serie danych i ich typ)
- wypisz pierwsze dwa wiersze
- policz ile jest wierszy dla każdego gatunku ("variety")
- oblicz średnią długość i szerokość płatków ("petal") dla każdego gatunku
- znajdź, który gatunek ma największą średnią długość płatka
- zapisz wyniki do pliku *wynik.csv*.
(`group_by`, `head`, `idxmax`, `info`, `mean`, `read_csv`, `to_csv`, `value_counts`)

In [ ]:
# do numpy - dobry do algebraicznych, problem z danymi rzeczywistymi - różne dane, nie rozróżnia 
# serie - tablice numpy, mamy inną 
# ramki (df) zbiór serii 
# indeksy - one są osobne, 
# dane czasowe - dobra obsługa - analizy, nie obliczenia 

df = pd.read_csv('iris.csv')
# wiersze
df.iloc[0:2, :]
df[:2]
# wartości dla kolumny 
df["variety"].value_counts()
# średnia długość 
df.groupby("variety")[['sepal.length', 'sepal.width']].mean().to_csv()
# najwieksza długość 
df.groupby("variety").mean()['sepal.length'].idxmax()
# Zapis do wyników 
#TODO 
df.




### Ćwiczenie 3
Poniżej znajdują się dane dotyczące ekspresji i liczby mutacji genów. 
- Wykonaj zestawienie danych obejmujące tylko geny, dla których mamy wszystkie informacje.
- Wykonaj zestawienie danych dla wszystkich genów, dla których mamy dane o ekspresji. 
- Wykonaj zestawienie dla wszystkich genów, dla których istnieją jakiekolwiek dane.
  
(`merge`)

In [ ]:
a = pd.DataFrame({
    'gen': ['BRCA1', 'TP53', 'EGFR'],
    'ekspresja': [120, 80, 60]
})

b = pd.DataFrame({
    'gen': ['TP53', 'EGFR', 'KRAS'],
    'liczba_mutacji': [3, 2, 5]
})

In [ ]:
a.merge(b, 'inner') # wszystkie, można też bez ale chodziło o intencjonalność 
a.merge(b, 'left') # ekspresja klucze względem lewej
a.merge(b, 'right') # liczba mutacji (względem prawej)
a.merge(b, 'outer') # wszystkie 

### Ćwiczenie 4
Dla poniższego zestawienia, dla każdej grupy genów oblicz:
- średnią ekspresję,
- maksymalną liczbę mutacji,
- liczbę genów w grupie.
Wynik zapisz do nowego DataFrame statystyki.

Następnie wybierz tylko te grupy, które mają:
- średnią ekspresję > 100,
- maksymalną liczbę mutacji ≥ 4.

(`agg`, `count`, `groupby`, `max`, `mean`, `query`)

In [ ]:
df = pd.DataFrame({
    'gen': ['BRCA1', 'BRCA2', 'TP53', 'EGFR', 'KRAS', 'MYC', 'NRAS', 'CDKN2A'],
    'grupa': ['DNA_repair', 'DNA_repair', 'Tumor_suppressor', 'Receptor', 'Receptor', 'Transcription', 'Receptor', 'Tumor_suppressor'],
    'ekspresja': [120, 130, 80, 60, 200, 250, 180, 90],
    'liczba_mutacji': [1, 2, 4, 2, 5, 0, 3, 6]
})

In [ ]:
# pierwsza część 
from pandas import NamedAgg


gr = df.groupby('grupa')
średnia = gr['ekspresja'].mean()
mutacje = gr['liczba_mutacji'].max()
count_genes = gr['gen'].count()
statystyki = pd.concat([średnia, mutacje, count_genes], axis=1)
statystyki

# druga część 
flag_1 = statystyki['ekspresja'] > 100 
flag_2 = statystyki['liczba_mutacji'] >= 4

statystyki
# statystyki[flag_1]
statystyki[flag_2]
statystyki[flag_2 & flag_1] 

# za pomocą agregacji 

gr.agg(
    mean_expression = pd.NamedAgg('ekspresja', 'mean'),
    max_mutations = pd.NamedAgg('liczba_mutacji', 'max'),
    count_genes = pd.NamedAgg('gen', 'count')
)


### Ćwiczenie 5

Dana jest ramka danych oraz lista zawierająca wszystkie unikatowe wartości jednej z kolumn ramki. Posortuj wiersze ramki tak, aby wartości w tej kolumnie występowały w tym samym porządku co na podanej liście.

Przykład:
Dla ramki danych

<table>
  <tr>
    <th>colA</th>
    <th>colB</th>
  </tr>
  <tr>
    <th>A</th>
    <th>1</th>
  </tr>
  <tr>
    <th>B</th>
    <th>2</th>
  </tr>
  <tr>
    <th>C</th>
    <th>3</th>
  </tr>
  <tr>
    <th>D</th>
    <th>4</th>
  </tr>
</table>

i listy ['C','A','D','B'] dla kolumny colA wynikiem powinno być

<table>
  <tr>
    <th>colA</th>
    <th>colB</th>
  </tr>
  <tr>
    <th>C</th>
    <th>3</th>
  </tr>
  <tr>
    <th>A</th>
    <th>1</th>
  </tr>
  <tr>
    <th>D</th>
    <th>4</th>
  </tr>
  <tr>
    <th>B</th>
    <th>2</th>
  </tr>
</table>

In [ ]:
df_ex2 = pd.DataFrame([["A", 1], ["B", 2], ["C", 3], ["D", 4]], columns = ["colA", "colB"])
df_ex2 

In [ ]:
order = {
    "C": 0,
    "A": 1,
    "D": 2,
    "B": 3
}

df_ex2.sort_values(by='colA', key=lambda x: x.map(order))

## Zadanie*

Poczytać o funkcji [pandas.pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html) i przeanalizować jej działanie w podanych przykładach. 

Następnie, dla danych podanych poniżej w ramce df_employees, za pomocą funkcji pivot_table utworzyć podsumowującą ramkę danych, której indeks będzie składał się z poszczególnych rodzajów zatrudnienia, tj. unikatowych wartości kolumny 'Type', a poszczególne kolumny będą zawierały sumę zarobków, wartość średnią zarobków i liczbę pracowników odzielnie dla każdego z wydziałów, tj. dla każdej z unikatowych wartości w kolumnie 'Department'. 

In [ ]:
import pandas as pd

df_employees = pd.DataFrame({'First Name': ['Tom', 'Adam', 'Jane', 'Alice', 'Robert', 'Bob', 'John', 'Frank', 'Eva',
                                            'John', 'Lois'],
                   'Last Name': ['Brown', 'Green', 'Thompson', 'Smith', 'Newman', 'Parker', 'Williams', 'Taylor', 'Evans',
                                 'Lewis', 'White'],
                   'Type': ['Full-time Employee', 'Intern', 'Full-time Employee', 'Part-time Employee', 
                            'Full-time Employee', 'Intern', 'Intern', 'Part-time Employee', 'Part-time Employee',
                            'Full-time Employee', 'Part-time Employee'],
                   'Department': ['Administration', 'Technical', 'Administration', 'Technical', 'Management',
                                  'Administration', 'Management', 'Administration', 'Management', 'Technical',
                                  'Management'],
                   'YearsOfExperience': [4, 3, 5, 7, 6, 1, 2, 2, 4, 5, 2],
                   'Salary': [15000, 6000, 9000, 10000, 20000, 3000, 4000, 6000, 12000, 8000, 5500]})

df_employees

In [ ]:
# indeks rodzaje zatrudnienia 'Type' / suma średnia ilość pracowników na wydział 'Department 
# pd.pivot_table(df_employees, values=['Type'], index=['Department'], columns=['Salary'], aggfunc={'Salary': 'mean', 'Salary': 'sum'})

pd.pivot_table(df_employees, values=['Salary'], index=['Type'], columns=['Department'], aggfunc=['mean', 'sum', 'count'])